To start the demo:
 - Open a terminal and run: docker compose -f docker-compose.cyborgdb.yaml up
 - Open a second terminal and run: docker compose -f docker-compose.op-inference.yaml up
 - You may have to wait for the servers to start running
 - Run All
 - At the bottom, open the url that ends with gradio.live

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "false"

%pip install -q langchain==0.3.26 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.3.1 \
    langchain-openai==0.3.28 \
    cryptography==39.0.1 \
    pypdf==5.8.0 \
    openai==1.97.1 \
    gradio==5.38.0 \
    einops==0.8.1 \
    sentence-transformers==4.1.0 \
    cyborgdb-0.11.1.dev26-py3-none-any.whl
    # cyborgdb==0.11.0


In [ ]:
import pathlib

import langchain.chains
import langchain.chains.combine_documents
import langchain.docstore.document
import langchain.document_loaders
import langchain.prompts
import langchain.text_splitter
import langchain_community.chat_message_histories
import langchain_core.runnables.history
import langchain_huggingface
import langchain_openai
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import base64

In [ ]:
EMBEDDING_MODEL_ID = "all-MiniLM-L6-v2"
MODEL_KWARGS = {"device": "cpu"}
CHAINS_STORE = {}
QUERY_ENCODE_KWARGS = None
CYBORGDB_KEYS: dict[str, str] = {}

In [ ]:
# Number of *characters*, not *tokens*, in a chunk
CHUNK_SIZE = 2048
# Number of *characters*, not *tokens*, to overlap between chunks
CHUNK_OVERLAP = 128

In [ ]:
def split_documents(
    file_path: str,
) -> list[langchain.docstore.document.Document]:
    """Split a document into smaller pieces for processing.

    LangChain has many different types of document loaders. For brevity, we will
    only use the CSVLoader, the PyPDFLoader, and the TextLoader.

    Args:
        file_path: Path to the uploaded file (specified by gradio).

    Returns:
        A list of documents, each containing a chunk of the original document.

    Raises:
        ValueError: If the file is not specified.
    """
    file = pathlib.Path(file_path)
    if not file or not file.exists():
        raise ValueError("File is required")

    # Switch over file types to determine how to load the document
    if file.suffix == ".csv":
        loader = langchain.document_loaders.CSVLoader(file)
        # There is no need to split the document if it is a CSV file, each
        # row will be treated as a separate document automatically.
        documents = loader.load()
    elif file.suffix == ".pdf":
        loader = langchain.document_loaders.PyPDFLoader(str(file))
        pages = loader.load()
        text_splitter = langchain.text_splitter.RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        documents = text_splitter.split_documents(pages)
    else:
        loader = langchain.document_loaders.TextLoader(file)
        pages = loader.load()
        text_splitter = langchain.text_splitter.RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        documents = text_splitter.split_documents(pages)

    return documents

In [ ]:
# We need a place to store the chat message histories and the chains for each user session.
# Keys are session IDs, values are the chat message histories and chains for that session.
CHAT_HISTORY_STORE: dict[
    str, langchain_community.chat_message_histories.ChatMessageHistory
] = {}
CHAINS_STORE: dict[
    str, langchain_core.runnables.history.RunnableWithMessageHistory
] = {}

In [ ]:
STAINED_GLASS_TRANSFORM_PROXY_PORT = 8600
STAINED_GLASS_TRANSFORM_PROXY_URL = (
    f"http://localhost:{STAINED_GLASS_TRANSFORM_PROXY_PORT}/v1"
)
STAINED_GLASS_TRANSFORM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"


In [ ]:
import cyborgdb as cyborgdb
from cyborgdb.integrations.langchain import CyborgVectorStore

CYBORGDB_HOST = "http://localhost:8001"
CYBORGDB_API_KEY = ""

In [ ]:

def encrypt_chunk(text: str, key: bytes) -> str:
    """Encrypt a text chunk using AES-256-GCM."""
    aesgcm = AESGCM(key)
    nonce = os.urandom(12)  # 12 bytes for GCM
    ciphertext = aesgcm.encrypt(nonce, text.encode(), None)
    # Combine nonce + ciphertext and encode as base64 for display
    encrypted_data = nonce + ciphertext
    return base64.b64encode(encrypted_data).decode()

# Replace your existing upload_and_create_cyborg_vector_store function with this enhanced version
def upload_and_create_cyborg_vector_store(
    file_paths: list[str],
    embedding_model_id: str,
    session_id: str,
    preview_chars: int = 64,  # Number of characters to show in preview
    max_previews: int = 10,     # Maximum number of encrypted chunks to preview
) -> str:
    """Upload files to CyborgDB vector store with encryption preview."""
    try:
        if not file_paths:
            return "No files selected"
        
        print(f"DEBUG: Processing {len(file_paths)} files for session {session_id}")
                    
        # Get or create the vector store for this session
        vectorstore = load_cyborgdb_vectorstore(session_id, embedding_model_id)
        
        # Get the encryption key for this session
        encryption_key = bytes(CYBORGDB_KEYS[session_id])
        
        # Process and add documents
        all_documents = []
        encrypted_previews = []
        
        for file_path in file_paths:
            print(f"DEBUG: Processing file: {file_path}")
            documents = split_documents(file_path)
            print(f"DEBUG: Split into {len(documents)} chunks")
            
            # Encrypt each chunk and create previews
            for i, doc in enumerate(documents):
                # Encrypt the chunk content
                encrypted_content = encrypt_chunk(doc.page_content, encryption_key)
                
                # Store preview of encrypted content (first N characters)
                if len(encrypted_previews) < max_previews:
                    preview = encrypted_content[:preview_chars]
                    encrypted_previews.append({
                        'file': os.path.basename(file_path),
                        'chunk_index': i,
                        'original_length': len(doc.page_content),
                        'encrypted_length': len(encrypted_content),
                        'encrypted_preview': preview
                    })
                
                print(f"DEBUG: Chunk {i}: Original={len(doc.page_content)} chars, Encrypted={len(encrypted_content)} chars")
            
            all_documents.extend(documents)
        
        if all_documents:
            print(f"DEBUG: Adding {len(all_documents)} documents to vector store")
            vectorstore.add_documents(all_documents)
            
            # Test retrieval immediately after adding
            print("DEBUG: Testing retrieval...")
            test_results = vectorstore.similarity_search("test", k=3)
            print(f"DEBUG: Retrieved {len(test_results)} documents for test query")
        
        # Create the response with encryption previews
        response_lines = [
            f"Successfully processed {len(file_paths)} files with {len(all_documents)} chunks",
            "",
        ]
        
        for i, preview in enumerate(encrypted_previews):
            response_lines.extend([
                f"Chunk {i+1}: {preview['encrypted_preview']}...",
                ""
            ])
        
        if len(all_documents) > max_previews:
            response_lines.append(f"... and {len(all_documents) - max_previews} more encrypted chunks")
        
        return "\n".join(response_lines)
        
    except Exception as e:
        print(f"DEBUG: Error in upload_and_create_cyborg_vector_store: {str(e)}")
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}"
    
def load_chain(
    session_id: str,
    system_prompt: str,
    embedding_model_id: str,
    openai_url: str | None,
    openai_model: str,
) -> langchain_core.runnables.history.RunnableWithMessageHistory:
    """Create a chain for retrieving answers to questions from CyborgDB and prompting the LLM."""

    print(f"DEBUG: Loading chain for session {session_id}")
            
    call_llm = langchain_openai.ChatOpenAI(
        base_url=openai_url, model=openai_model, max_tokens=500, temperature=0.7
    )
    
    # Get the CyborgDB vector store for this session
    vectordb = load_cyborgdb_vectorstore(session_id, embedding_model_id)
    
    # Test retrieval before creating retriever
    print("DEBUG: Testing vector store retrieval in chain...")
    test_docs = vectordb.similarity_search("test", k=1)
    print(f"DEBUG: Vector store has {len(test_docs)} documents for test query")
    
    retriever = vectordb.as_retriever(search_kwargs={"k": 3})

    # Ask the LLM to reformulate user prompts as queries for the document store
    contextualize_question_system_prompt = (
        "Given a chat history and the latest user question "
        "which might reference context in the chat history, formulate a standalone question "
        "which can be understood without the chat history. If the question is not related to the chat history, "
        "leave the question intact. Do NOT answer the question, "
        "just reformulate it if needed and otherwise return it as is."
    )
    contextualize_question_prompt = (
        langchain.prompts.ChatPromptTemplate.from_messages(
            [
                ("system", contextualize_question_system_prompt),
                langchain.prompts.MessagesPlaceholder("chat_history"),
                ("human", "{input}"),
            ]
        )
    )

    # Retrieve the most relevant documents for a given question from the vector database
    history_aware_retriever = langchain.chains.create_history_aware_retriever(
        call_llm, retriever, contextualize_question_prompt
    )

    # Create a new prompt including the chat history and the retrieved documents
    document_prompt = langchain.prompts.PromptTemplate(
        input_variables=["page_content", "source"],
        template="Context:\n{page_content}\n\nSource: {source}",
    )
    user_chat_prompt = langchain.prompts.ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            langchain.prompts.MessagesPlaceholder("chat_history"),
            ("human", "{input}"),
        ]
    )

    # Query the LLM with the chat history and the most relevant documents
    user_chat_chain_with_documents = (
        langchain.chains.combine_documents.create_stuff_documents_chain(
            call_llm, user_chat_prompt, document_prompt=document_prompt
        )
    )

    # Compose the retrieval and chat chains
    rag_chain = langchain.chains.create_retrieval_chain(
        history_aware_retriever, user_chat_chain_with_documents
    )

    # Create a chain that stores the chat message history for the session, using the RAG chain
    conversation_rag_chain = (
        langchain_core.runnables.history.RunnableWithMessageHistory(
            rag_chain,
            get_session_history=lambda: CHAT_HISTORY_STORE.get(
                session_id,
                langchain_community.chat_message_histories.ChatMessageHistory(),
            ),
            input_messages_key="input",
            history_messages_key="chat_history",
            output_messages_key="answer",
        )
    )

    print("DEBUG: Chain loaded successfully")
    return conversation_rag_chain

def load_cyborgdb_vectorstore(
    session_id: str,
    embedding_model_id: str,
) -> CyborgVectorStore:
    """Load a CyborgDB vector store with debugging."""
    try:
        # Generate a unique index key for this session
        if session_id not in CYBORGDB_KEYS:
            index_key = cyborgdb.generate_key()
            CYBORGDB_KEYS[session_id] = index_key
            print(f"DEBUG: Generated new index key for session {session_id}")
        else:
            index_key = CYBORGDB_KEYS[session_id]
            print(f"DEBUG: Using existing index key for session {session_id}")
        
        # Create the vector store - it will automatically create the index if needed
        vector_store = CyborgVectorStore(
            index_name=f"session_{session_id}",
            index_key=index_key,
            api_key=CYBORGDB_API_KEY,
            api_url=CYBORGDB_HOST,
            embedding=embedding_model_id,
            index_type="ivfflat",
            metric="cosine",
        )
        
        print(f"DEBUG: Created CyborgVectorStore for session {session_id}")
        return vector_store
    except Exception as e:
        print(f"DEBUG: Error creating vector store: {str(e)}")
        import traceback
        traceback.print_exc()
        raise

In [ ]:
import uuid

import gradio as gr
import requests

MOST_RECENT_PROMPT: str | None = None


def fetch_session_hash(request: gr.Request) -> str | None:
    """Fetch the session hash from the request.

    We will use the session hash as the CyborgDB collection name so that each
    individual session has its own collection in CyborgDB.

    Args:
        request: The Gradio request object.

    Returns:
        The session hash.
    """
    return request.session_hash


def chat_bot(
    message: str,
    history: list[list[str]],
    session: str,
    system_prompt: str,
    openai_url: str | None,
    openai_model: str,
    embedding_model_id: str,
) -> str:
    """Chat with the RAG conversation agent.

    Args:
        message: The message from the user.
        history: The chat history. Unused, because langchain handles its own copy of the chat history.
        session: The session hash.
        system_prompt: The system prompt.
        openai_url: The URL of the OpenAI API.
        openai_model: The OpenAI model to use for answering questions.
        embedding_model_id: The HuggingFace model ID to use for embedding the documents.

    Returns:
        The response from the RAG conversation agent.
    """
    global MOST_RECENT_PROMPT
    MOST_RECENT_PROMPT = message

    if session not in CHAINS_STORE:
        CHAINS_STORE[session] = load_chain(
            session,
            system_prompt,
            embedding_model_id,  # Fixed: removed duplicate session parameter
            openai_url,
            openai_model,
        )
    chain = CHAINS_STORE[session]
    response = chain.invoke(
        {"input": message}, config={"configurable": {"session_id": session}}
    )
    return response["answer"]


def see_stainedglass_reconstruction(openai_url: str) -> str:
    """Reconstruct the most recent prompt using the StainedGlass API.

    Args:
        openai_url: The URL of the OpenAI API.

    Returns:
        The reconstructed prompt.
    """
    if MOST_RECENT_PROMPT is None:
        return "No prompt to reconstruct"

    request = {
        "messages": [
            {"role": "user", "content": MOST_RECENT_PROMPT},
        ],
        "return_transformed_embeddings": False,
        "return_reconstructed_prompt": True,
        "return_plain_text_embeddings": False,
    }
    try:
        response = requests.post(f"{openai_url}/stainedglass", json=request).json()
        return response["reconstructed_prompt"]
    except Exception as e:
        return f"Error reconstructing prompt: {str(e)}"


def setup_cyborg_chatbot_web_app(
    openai_url: str | None,
    openai_model: str,
    show_reconstructed_prompt: bool = False,
) -> gr.Blocks:
    """Set up the Gradio interface for the RAG conversation agent.

    Args:
        cyborg_host: The host of the CyborgDB server.
        openai_url: The URL of the OpenAI API.
        openai_model: The OpenAI model to use for answering questions.
        show_reconstructed_prompt: Whether to show the reconstructed prompt.

    Returns:
        The Gradio interface for the RAG conversation agent.
    """
    with gr.Blocks() as demo, gr.Row():
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("## RAG Conversation Agent with CyborgDB + Protopia AI")
            system_prompt = gr.Textbox(
                label="System instruction",
                lines=3,
                value="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Keep the answer concise. {context}",
            )

            session = gr.Textbox(value=str(uuid.uuid4()), label="Session")  # Fixed: convert to string
            chat_interface = gr.ChatInterface(
                fn=lambda message, history, session, system_prompt: chat_bot(
                    message,
                    history,
                    session,
                    system_prompt,
                    openai_url,
                    openai_model,
                    EMBEDDING_MODEL_ID,
                ),
                additional_inputs=[
                    session,
                    system_prompt,
                ],
            )
            demo.load(fetch_session_hash, None, session)

            if show_reconstructed_prompt:
                gr.Markdown(
                    "### Attempted Text Reconstruction of Most Recent Prompt Protected by Protopia Stained Glass Transform"
                )
                most_recent_prompt = gr.Textbox(
                    MOST_RECENT_PROMPT,
                    lines=3,
                    placeholder="Attempted text reconstruction...",
                    label="",
                )
                if openai_url is not None:
                    chat_interface.textbox.submit(
                        lambda: see_stainedglass_reconstruction(openai_url),
                        None,
                        most_recent_prompt,
                        trigger_mode="always_last",
                    )

        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("## Upload Document")
            file = gr.File(type="filepath", file_count="multiple")
            with gr.Row(equal_height=True), gr.Column(variant="compact"):
                create_vector_store_button = gr.Button(
                    "Create vector store", variant="primary", scale=1
                )
                vector_index_msg_out = gr.Textbox(
                    show_label=False,
                    lines=1,
                    scale=1,
                    placeholder="Please create vector store...",
                )

            create_vector_store_button.click(
                lambda files, session_id: upload_and_create_cyborg_vector_store(
                    files if files else [],
                    EMBEDDING_MODEL_ID,
                    session_id,
                ),
                inputs=[file, session],
                outputs=[vector_index_msg_out],
            )
            gr.HTML(
                """
                <style>
                    #my_image img {
                        border: 1px solid #e4e4e7;  /* green border, change color as needed */
                        border-radius: 8px;         /* rounded corners */
                        padding: 4px;               /* space between border and image */
                        background-color: none;    /* optional background */
                    }
                </style>
                """
            )
            gr.Image(
                value='Cyborg-Protopia Diagrams (Simplified) 2.png',
                show_label=False,
                interactive=False,
                height=280,      # tweak as you like, or remove
                elem_id="my_image"
            )

        return demo

In [ ]:
OPENAI_URL = None
OPENAI_MODEL = "gpt-4o-mini"

# # You can set the OPENAI_API_KEY environment variable using the lines below.
import os
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
protected_demo = setup_cyborg_chatbot_web_app(
    openai_url=STAINED_GLASS_TRANSFORM_PROXY_URL,
    openai_model=STAINED_GLASS_TRANSFORM_MODEL,
    show_reconstructed_prompt=True,
)

protected_demo.launch(inline=False, share=True)